# Binary Classification with Neural Networks

<a href="https://colab.research.google.com/github/HassanAlgoz/dl/blob/main/modules/deep-learning/02-lab_nn/lab_nn_classification.ipynb" target="_blank">
  <img src="https://raw.githubusercontent.com/HassanAlgoz/dl/main/assets/Open%20in%20Colab-F9AB00.svg" alt="Open in Colab" height="50"/>
</a>

Train a neural network to separate two classes in 2D.

You'll generate points whose label depends on the **sum of two features**, fit an `MLPClassifier`, and plot the **decision boundary** — the line the model draws between class 0 and class 1.


In [ ]:
# --- Setup: Clone repo & cd into correct folder (Colab only) ---
import os
import sys
import subprocess

if "google.colab" in sys.modules:
    repo_url = "https://github.com/HassanAlgoz/dl.git"
    lab_folder = "dl/modules/deep-learning/02-lab_nn"

    # Only clone if the folder doesn't exist
    if not os.path.exists(lab_folder):
        subprocess.run(["git", "clone", repo_url])

    # Change working directory to the lab folder
    os.chdir(lab_folder)


## Imports

- **`pandas`**: hold features in a DataFrame with named columns
- **`numpy`**: generate the synthetic 2D points
- **`Pipeline`**: chain preprocessing and the model so they stay in sync
- **`StandardScaler`**: standardize features (mean 0, variance 1)
- **`MLPClassifier`**: a neural network for predicting a class label

This lab predicts a class (0 or 1), so you'll use [`MLPClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html). For a continuous target, scikit-learn has [`MLPRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html).


In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

import helper_utils


## The Classification Problem

Each point has two features. The true label is **1** when `feature_1 + feature_2 > 0`, and **0** otherwise.

That rule is a straight diagonal cut through the plane — linearly separable. A small network should recover a similar boundary.


## Generate Data

Build 100 random 2D points.

* `X`: a DataFrame with two columns, `feature_1` and `feature_2` (2D — what scikit-learn expects for features)
* `y`: class labels 0 or 1 (1D — what `MLPClassifier` expects)


In [ ]:
rng = np.random.default_rng(42)
raw = rng.normal(size=(100, 2))

X = pd.DataFrame(raw, columns=["feature_1", "feature_2"])
y = (X["feature_1"] + X["feature_2"] > 0).astype(int)

X.head()


In [ ]:
y.head()


The points are colored by class. You should see two groups split along the diagonal `feature_1 + feature_2 = 0`.


In [ ]:
helper_utils.plot_data(X, y, title="Binary Classification Data")


## Architecture

A fully connected net with 2 inputs and 4 hidden ReLU neurons would be `2 → 4 (ReLU) → output`. In scikit-learn you mainly choose the **hidden layer** and the **activation** — `MLPClassifier` adds the output layer.

* **`hidden_layer_sizes=(4,)`**: one hidden layer with four neurons. Each computes a weighted sum of the two features, then applies ReLU.
* **`activation="relu"`**: ReLU on each hidden neuron, so the decision boundary can bend if it needs to.
* **Output layer**: added for you. For **binary** classification that's **one** neuron with a logistic (sigmoid) activation — not two softmax units. The predicted class is 1 when that probability is above 0.5.
* **`solver="lbfgs"`**: a good fit for a hundred rows. Often more reliable than Adam or SGD on small data.

Flow: scaled features → 4 hidden neurons → ReLU → one probability → class 0 or 1.


### Scaling Features in a Pipeline

Neural networks are sensitive to input scale. These two features are already standard-normal, but putting a `StandardScaler` in a **pipeline** is the habit you want: on `fit`, the scaler learns from the training features, then the MLP trains on scaled values. On `predict`, the same fitted scaler is applied automatically.


In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(4,),
        activation="relu",
        solver="lbfgs",
        max_iter=1000,
        random_state=42,
    )),
])


## Training

Training is a single call: `model.fit(X, y)`.

`MLPClassifier` minimizes **log-loss** (cross-entropy) — how surprised the model is by the true labels — and updates the weights. You don't write a training loop yourself.


In [ ]:
model.fit(X, y)

print("Training complete.")


## Visualize the Decision Boundary

Color the plane by the predicted class. The boundary should cut near the diagonal that generated the labels.


In [ ]:
helper_utils.plot_decision_boundary(
    model,
    X,
    y,
    title="Decision Boundary (ReLU, 4 hidden)",
)


## Predict

Classify a new point. Pass a one-row DataFrame with the same column names used in training.


In [ ]:
new_point = pd.DataFrame({"feature_1": [1.5], "feature_2": [0.5]})
predicted_class = model.predict(new_point)[0]
predicted_proba = model.predict_proba(new_point)[0]

print(f"Predicted class: {predicted_class}")
print(f"Class probabilities: class 0 = {predicted_proba[0]:.2f}, class 1 = {predicted_proba[1]:.2f}")


## Conclusion

See scikit-learn's [neural network models (supervised)](https://scikit-learn.org/stable/modules/neural_networks_supervised.html) guide and the [`MLPClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html) docs.
